<a href="https://colab.research.google.com/github/Windy0628/PET-Image-Reconstructions-FBP-vs-Iterative-Reconstruction-MLEM-OSEM-/blob/main/notebooks/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import binary_erosion
from skimage.data import shepp_logan_phantom
from skimage.transform import radon, iradon, resize

Step1. Build Shepp-Logan phantom.

In [20]:
N = 128
COUNTS = None
FILTERS = ["ramp", "shepp-logan", "cosine", "hamming", "hann"]
C = N // 2 # Center for plotting profiles
phantom = resize(shepp_logan_phantom(),(N, N))
yy, xx = np.mgrid[:N, :N] - N // 2
phantom[xx**2 + yy**2 > (N // 2) ** 2] = 0

Step2. Build sinogram

In [21]:
theta = np.linspace(0., 180., 180, endpoint=False)
sinogram = radon(phantom, theta=theta, circle=True)

A.step1's diagram

In [22]:
def fov_mask(n=N):
    """True inside the circular field of view (same circle skimage.radon assumes)."""
    yy, xx = np.mgrid[:n, :n] - n // 2
    return xx**2 + yy**2 <= (n // 2) ** 2

In [23]:
def make_phantom(n=N):
    img = resize(shepp_logan_phantom(), (n, n))
    img[~fov_mask(n)] = 0
    return img

In [24]:
def simulate(truth, counts=None, rng=None):
    """Sinogram of `truth`. counts=None -> noise-free; otherwise Poisson noise with
    `counts` total events, rescaled back to the noise-free units."""
    sino = radon(truth, theta=theta, circle=True)
    if counts is None:
        return sino
    rng = rng or np.random.default_rng(0)
    k = counts / sino.sum()
    return rng.poisson(sino * k) / k

In [25]:
def plot_phantom_sinogram(phantom, sinogram, fname="phantom_sinogram.png"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.8))

    im1 = ax1.imshow(phantom, cmap="gray")
    ax1.set_title("Shepp-Logan phantom")
    ax1.set_xlabel("x (pixel)"); ax1.set_ylabel("y (pixel)")
    fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04, label="Activity (a.u.)")

    n_det = sinogram.shape[0]
    d_theta = theta[1] - theta[0]
    s_min, s_max = -(n_det // 2), n_det - 1 - n_det // 2
    im2 = ax2.imshow(sinogram, cmap="gray", aspect="auto",
                     extent=(theta[0] - d_theta / 2, theta[-1] + d_theta / 2,  # left, right
                             s_max + 0.5, s_min - 0.5))
    ax2.set_title("Sinogram (Radon transform)")
    ax2.set_xlabel("Projection angle θ (degrees)")
    ax2.set_ylabel("Detector position s (pixel)")
    fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04, label="Line integral (a.u.)")

    plt.tight_layout()
    plt.savefig(fname, dpi=150)
    plt.close(fig)

B.  FBP reconstruction

Perform FBP reconstruction using each of the five filters

In [26]:
def fbp(sino, filter_name):
  img = iradon(sino, theta=theta, filter_name=filter_name,
                 circle=True, output_size=N)
  img[~fov_mask()] = 0
  return img

In [27]:
def make_rois(n=N, erode=3):
    """
    Build ROIs from the piecewise-constant phantom (nearest-neighbour resize,
    so each region has one exact value), then erode them by `erode` pixels so
    that partial-volume edge pixels are excluded.
      hot : the 0.3 ellipse in the upper half (true contrast vs. background = 0.5)
      bkg : the 0.2 'brain' region (large and uniform)
    """
    lab = np.round(resize(shepp_logan_phantom(), (n, n), order=0,
                          anti_aliasing=False), 1)
    rows = np.arange(n)[:, None] * np.ones((1, n))
    hot = (lab == 0.3) & (rows < n // 2)
    bkg = lab == 0.2
    st = np.ones((3, 3), bool)
    return (binary_erosion(hot, st, iterations=erode),
            binary_erosion(bkg, st, iterations=erode))

In [28]:
def nrmse(recon, truth, mask=None):
    """RMSE / RMS(truth) inside `mask` (default: field of view). 0 = perfect."""
    m = fov_mask() if mask is None else mask
    return np.sqrt(np.mean((recon[m] - truth[m]) ** 2)) / np.sqrt(np.mean(truth[m] ** 2))


def contrast_recovery(recon, truth, hot, bkg):
    """CR = (hot/bkg - 1)_recon / (hot/bkg - 1)_truth. 1 = contrast fully recovered."""
    c_rec = recon[hot].mean() / recon[bkg].mean() - 1
    c_true = truth[hot].mean() / truth[bkg].mean() - 1
    return c_rec / c_true


def uniform_std(recon, bkg):
    """Standard deviation in the uniform region (same units as the image).
    Noise-free data: measures artefacts (ringing/streaks). Noisy data: measures noise."""
    return recon[bkg].std()


def evaluate(recon, truth, hot, bkg):
    sd = uniform_std(recon, bkg)
    return {"NRMSE": nrmse(recon, truth),
            "CR": contrast_recovery(recon, truth, hot, bkg),
            "SD_bkg": sd,
            "COV_bkg": sd / recon[bkg].mean(),      # relative noise
            "bkg_mean": recon[bkg].mean()}


def plot_fbp_eval(truth, recons, results, hot, bkg, title, fname):
    fig, ax = plt.subplots(2, 6, figsize=(18, 6.4))

    ax[0, 0].imshow(truth, cmap="gray", vmin=0, vmax=1)
    ax[0, 0].contour(hot, levels=[0.5], colors="r", linewidths=1)
    ax[0, 0].contour(bkg, levels=[0.5], colors="c", linewidths=1)
    ax[0, 0].set_title("Truth + ROIs\n(red: hot, cyan: uniform)")

    s = np.arange(N) - C
    ax[1, 0].plot(s, truth[:, C], "k", lw=2, label="Truth")
    for f in ["ramp", "hann"]:
        ax[1, 0].plot(s, recons[f][:, C], lw=1, label=f)
    ax[1, 0].set_title("Vertical profile through centre")
    ax[1, 0].set_xlabel("y (pixel)"); ax[1, 0].set_ylim(-0.1, 1.1)
    ax[1, 0].legend(fontsize=8)

    for j, f in enumerate(FILTERS, start=1):
        r = results[f]
        ax[0, j].imshow(recons[f], cmap="gray", vmin=0, vmax=1)
        ax[0, j].set_title(f"{f}\nNRMSE={r['NRMSE']:.3f}  CR={r['CR']:.2f}")
        im = ax[1, j].imshow(recons[f] - truth, cmap="RdBu_r", vmin=-0.3, vmax=0.3)
        ax[1, j].set_title(f"Error (recon - truth)\nSD_bkg={r['SD_bkg']:.4f}")

    for a in ax.flat:
        if a is not ax[1, 0]:
            a.axis("off")
    fig.colorbar(im, ax=ax[1, 1:].tolist(), fraction=0.02, pad=0.01, label="Error")
    fig.suptitle(title, fontsize=13)
    plt.savefig(fname, dpi=110, bbox_inches="tight")
    plt.close(fig)



Main

In [29]:
if __name__ == "__main__":
    # ----- Part A -----
    phantom = make_phantom()
    sinogram = simulate(phantom, counts=COUNTS)
    plot_phantom_sinogram(phantom, sinogram)
    print(f"Phantom shape:  {phantom.shape}")
    print(f"Sinogram shape: {sinogram.shape}  (detector bins x angles)")
    print("Figure saved to phantom_sinogram.png\n")

    # ----- Part B -----
    hot, bkg = make_rois()
    print(f"Hot ROI: {hot.sum()} px, uniform ROI: {bkg.sum()} px")
    print(f"Truth: hot mean = {phantom[hot].mean():.3f}, "
          f"uniform mean = {phantom[bkg].mean():.3f}\n")

    recons, results = {}, {}
    print(f"{'filter':12s} {'NRMSE':>7s} {'CR':>7s} {'SD_bkg':>8s} {'COV_bkg':>8s} {'bkg mean':>9s}")
    for f in FILTERS:
        recons[f] = fbp(sinogram, f)
        results[f] = r = evaluate(recons[f], phantom, hot, bkg)
        print(f"{f:12s} {r['NRMSE']:7.3f} {r['CR']:7.3f} {r['SD_bkg']:8.4f} "
              f"{r['COV_bkg']:8.3f} {r['bkg_mean']:9.3f}")

    tag = "noisefree" if COUNTS is None else f"counts{int(COUNTS)}"
    title = ("FBP filters on noise-free data" if COUNTS is None
             else f"FBP filters, Poisson noise ({COUNTS:.0e} counts)")
    plot_fbp_eval(phantom, recons, results, hot, bkg, title, f"fbp_eval_{tag}.png")
    print(f"\nFigure saved to fbp_eval_{tag}.png")

Phantom shape:  (128, 128)
Sinogram shape: (128, 180)  (detector bins x angles)
Figure saved to phantom_sinogram.png

Hot ROI: 330 px, uniform ROI: 3183 px
Truth: hot mean = 0.298, uniform mean = 0.200

filter         NRMSE      CR   SD_bkg  COV_bkg  bkg mean
ramp           0.136   1.001   0.0029    0.014     0.200
shepp-logan    0.158   1.001   0.0022    0.011     0.200
cosine         0.203   1.001   0.0012    0.006     0.200
hamming        0.238   1.001   0.0009    0.005     0.200
hann           0.247   1.001   0.0008    0.004     0.200

Figure saved to fbp_eval_noisefree.png
